In [1]:
# ============================================================
# notebooks/07_contextual_features.py  — INDUSTRY VERSION
#
# WHAT CHANGED FROM PREVIOUS VERSION:
#   1. Labels: replaced isFraud (transaction-level) with
#      account_compromised (account-window label).
#      Marks transactions 24h BEFORE first fraud as positive.
#      Teaches the model what behavior looks like DURING takeover,
#      not just at the confirmed fraud transaction.
#
#   2. Features: expanded from 8 to 13.
#      New features: amount_percentile, time_gap_norm,
#      merchant_fraud_rate, cross_device_accounts, account_age_score.
#
# OUTPUTS:
#   data/contextual_features.npy     — shape (N, 13), float32
#   data/fusion_labels.npy           — account_compromised labels
#   data/fusion_account_ids.npy      — card1 per row
#   data/fusion_feature_names.json   — 13 feature names
#   data/drift_norm_params.json      — autoencoder normalisation
#   data/merchant_fraud_rates.json   — precomputed per ProductCD
#
# RUNTIME: ~20 min
# ============================================================



In [4]:
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

ROOT = os.path.abspath("..")

DATA_DIR  = os.path.join(ROOT, "data")
MODEL_DIR = os.path.join(ROOT, "models")
SNAP_PATH = os.path.join(DATA_DIR, "tx_snapshot.parquet")

print("ROOT =", ROOT)



ROOT = f:\rxtj_phase_2


In [5]:
print("\n[NB07] Loading snapshot...")
tx = pd.read_parquet(SNAP_PATH)
print(f"  {len(tx):,} rows  |  {len(tx.columns)} cols")
print(f"  isFraud rate: {tx['isFraud'].mean()*100:.2f}%")




[NB07] Loading snapshot...
  590,540 rows  |  15 cols
  isFraud rate: 3.50%


In [6]:
print("\n[NB07] Loading Phase 1 artifacts...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    imputer = joblib.load(os.path.join(MODEL_DIR, "imputer.pkl"))
    scaler  = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
RAW_DIM = int(imputer.n_features_in_)

class _AE(nn.Module):
    def __init__(self, d, lat=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, lat))
        self.decoder = nn.Sequential(
            nn.Linear(lat, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, d))
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z), z

ae_state = torch.load(os.path.join(MODEL_DIR, "autoencoder.pt"), map_location="cpu")
ae_in    = ae_state["encoder.0.weight"].shape[1]
ae       = _AE(ae_in)
ae.load_state_dict(ae_state, strict=False)
ae.eval()
print(f"  imputer: {RAW_DIM}  |  autoencoder input: {ae_in}")




[NB07] Loading Phase 1 artifacts...
  imputer: 224  |  autoencoder input: 224


In [7]:
print("\n[NB07] Computing behavioral drift scores...")
pcd_map = {"W": 0, "H": 1, "C": 2, "S": 3, "R": 4}
X_raw   = np.full((len(tx), RAW_DIM), np.nan, dtype=np.float32)
X_raw[:, 0] = tx["TransactionAmt"].fillna(0).values.astype(np.float32)
X_raw[:, 1] = tx["hour"].values.astype(np.float32)
X_raw[:, 2] = tx["ProductCD"].map(pcd_map).fillna(-1).values.astype(np.float32)
X_raw[:, 3] = pd.to_numeric(tx["addr1"], errors="coerce").values.astype(np.float32)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    X_sc = scaler.transform(imputer.transform(X_raw)).astype(np.float32)

BATCH        = 2048
recon_errors = []
with torch.no_grad():
    for s in range(0, len(X_sc), BATCH):
        b = torch.FloatTensor(X_sc[s:s+BATCH])
        r, _ = ae(b)
        recon_errors.extend(((r-b)**2).mean(dim=1).cpu().numpy().tolist())
        if (s // BATCH) % 50 == 0:
            print(f"  {min(s+BATCH,len(X_sc)):>7,}/{len(X_sc):,}")

recon = np.array(recon_errors, dtype=np.float32)
p5, p95 = np.percentile(recon, 5), np.percentile(recon, 95)
f7_drift = np.clip((recon - p5) / (p95 - p5 + 1e-9), 0.0, 1.0).astype(np.float32)
with open(os.path.join(DATA_DIR, "drift_norm_params.json"), "w") as f:
    json.dump({"p5": float(p5), "p95": float(p95)}, f)
print(f"  drift range: {recon.min():.4f}–{recon.max():.4f}  "
      f"p5={p5:.4f}  p95={p95:.4f}")




[NB07] Computing behavioral drift scores...
    2,048/590,540
  104,448/590,540
  206,848/590,540
  309,248/590,540
  411,648/590,540
  514,048/590,540
  drift range: 0.1995–0.2530  p5=0.2056  p95=0.2463


In [8]:
print("\n[NB07] Computing contextual features (F1–F6)...")

# F1: amount z-score
cum_std = np.where(tx["amt_cumstd"].fillna(0).values < 0.1, 1.0,
                   tx["amt_cumstd"].values.astype(np.float32))
f1_amount_z = np.clip(
    (tx["TransactionAmt"].fillna(0).values.astype(np.float32)
     - tx["amt_cumsum"].fillna(0).values.astype(np.float32)) / cum_std,
    -5.0, 5.0).astype(np.float32)
print(f"  F1 amount_z_score   mean={f1_amount_z.mean():.3f}")

# F2: merchant novelty
tx["pcd_int"] = tx["ProductCD"].map({"W":0,"H":1,"C":2,"S":3,"R":4}).fillna(-1).astype(int)

def _pcd_novelty(g):
    pcd, nov, cnt = g["pcd_int"].values, np.ones(len(g), np.float32), {}
    for j, p in enumerate(pcd):
        if j > 0 and p in cnt: nov[j] = 1.0 - cnt[p]/j
        cnt[p] = cnt.get(p, 0) + 1
    return pd.Series(nov, index=g.index)

f2_merchant = tx.groupby("card1", group_keys=False).apply(_pcd_novelty).values.astype(np.float32)
print(f"  F2 merchant_novelty mean={f2_merchant.mean():.3f}")

# F3: geo displacement (float addr1 expanding median)
tx2 = tx.copy()
tx2["addr1_f"] = pd.to_numeric(tx2["addr1"], errors="coerce").fillna(0.0)
tx2["addr1_baseline"] = (tx2.groupby("card1")["addr1_f"]
                          .transform(lambda s: s.expanding().median().shift(1))
                          .fillna(tx2["addr1_f"]))
f3_geo = np.clip(np.abs(tx2["addr1_f"].values - tx2["addr1_baseline"].values) / 500.0,
                 0.0, 1.0).astype(np.float32)
f3_geo = np.nan_to_num(f3_geo, nan=0.0)
print(f"  F3 geo_displacement mean={f3_geo.mean():.3f}  NaN={np.isnan(f3_geo).sum()}")

# F4: hour deviation
def _hour_dev(g):
    hrs, dev, cnt = g["hour"].values, np.zeros(len(g), np.float32), [0]*24
    for j, h in enumerate(hrs):
        if j > 0: dev[j] = 1.0 - cnt[int(h)]/j
        cnt[int(h)] += 1
    return pd.Series(dev, index=g.index)

f4_hour = tx.groupby("card1", group_keys=False).apply(_hour_dev).values.astype(np.float32)
print(f"  F4 hour_deviation   mean={f4_hour.mean():.3f}")

# F5: device novelty
tx["device_str"] = tx["DeviceInfo"].fillna("").astype(str)
tx["device_key"] = tx["card1"].astype(str) + "|||" + tx["device_str"]
first_seen = (tx.groupby("device_key")["TransactionDT"].min()
                .rename("first_dt").reset_index())
tx = tx.merge(first_seen, on="device_key", how="left")
f5_device = np.where(tx["device_str"] == "", 0.5,
            np.where(tx["TransactionDT"] == tx["first_dt"], 1.0, 0.0)
            ).astype(np.float32)
print(f"  F5 device_novelty   mean={f5_device.mean():.3f}")

# F6: velocity ratio
vel_1h  = tx["velocity_1h"].fillna(0).values.astype(np.float32)
vel_24h = tx["velocity_24h"].fillna(0).values.astype(np.float32)
f6_vel  = np.clip(vel_1h / (vel_24h/24.0 + 1e-6), 0.0, 10.0).astype(np.float32)
print(f"  F6 velocity_ratio   mean={f6_vel.mean():.3f}")




[NB07] Computing contextual features (F1–F6)...
  F1 amount_z_score   mean=0.026


C:\Users\sweth\AppData\Local\Temp\ipykernel_26260\3851328815.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  f2_merchant = tx.groupby("card1", group_keys=False).apply(_pcd_novelty).values.astype(np.float32)


  F2 merchant_novelty mean=0.231
  F3 geo_displacement mean=0.097  NaN=0


C:\Users\sweth\AppData\Local\Temp\ipykernel_26260\3851328815.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  f4_hour = tx.groupby("card1", group_keys=False).apply(_hour_dev).values.astype(np.float32)


  F4 hour_deviation   mean=0.912
  F5 device_novelty   mean=0.444
  F6 velocity_ratio   mean=1.595


In [9]:
f8_p1 = np.full(len(tx), 0.5, dtype=np.float32)
print(f"  F8 p1_risk_score    placeholder 0.5 (NB08 will patch)")



  F8 p1_risk_score    placeholder 0.5 (NB08 will patch)


In [10]:
# Where does this transaction's amount rank in account's historical distribution?
# 0 = lowest ever, 1 = highest ever. Better than z-score for skewed distributions.
print("\n[NB07] Computing new features (F9–F13)...")
tx["txn_seq"] = tx.get("txn_seq", tx.groupby("card1").cumcount())
tx["amt_rank"] = (tx.groupby("card1")["TransactionAmt"]
                   .transform(lambda s: s.expanding().rank(pct=True).shift(1))
                   .fillna(0.5))
f9_amt_pct = tx["amt_rank"].values.astype(np.float32)
print(f"  F9  amount_percentile  mean={f9_amt_pct.mean():.3f}")




[NB07] Computing new features (F9–F13)...
  F9  amount_percentile  mean=0.529


In [11]:
# Large gaps before a burst of transactions = dormant account being used by attacker.
# Log-normalised so large gaps don't dominate.
def _time_gap(group):
    dt   = group["TransactionDT"].values.astype(np.float64)
    gaps = np.zeros(len(dt), dtype=np.float32)
    for j in range(1, len(dt)):
        hours      = (dt[j] - dt[j-1]) / 3600.0
        gaps[j]    = float(np.log1p(max(hours, 0.0)))
    return pd.Series(gaps, index=group.index)

f10_gap = tx.groupby("card1", group_keys=False).apply(_time_gap).values.astype(np.float32)
# Normalise to [0,1]
gap_max = np.percentile(f10_gap, 99)
f10_gap = np.clip(f10_gap / (gap_max + 1e-9), 0.0, 1.0).astype(np.float32)
print(f"  F10 time_gap_norm      mean={f10_gap.mean():.3f}")



  F10 time_gap_norm      mean=0.243


C:\Users\sweth\AppData\Local\Temp\ipykernel_26260\1723569408.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  f10_gap = tx.groupby("card1", group_keys=False).apply(_time_gap).values.astype(np.float32)


In [12]:
# What fraction of ALL transactions at this merchant category are fraud?
# Pre-computed from training data — a high-risk merchant = extra signal.
mfr = (tx.groupby("ProductCD")["isFraud"]
         .mean().rename("mfr").reset_index())
tx = tx.merge(mfr, on="ProductCD", how="left")
f11_mfr = tx["mfr"].fillna(tx["isFraud"].mean()).values.astype(np.float32)
# Save for inference time
mfr_dict = dict(zip(mfr["ProductCD"], mfr["mfr"].round(6)))
with open(os.path.join(DATA_DIR, "merchant_fraud_rates.json"), "w") as f:
    json.dump(mfr_dict, f, indent=2)
print(f"  F11 merchant_fraud_rate mean={f11_mfr.mean():.4f}  "
      f"values={mfr_dict}")



  F11 merchant_fraud_rate mean=0.0350  values={'C': 0.116873, 'H': 0.047662, 'R': 0.037826, 'S': 0.058996, 'W': 0.020399}


In [13]:
# How many DIFFERENT accounts (card1) have used this device?
# A device shared across 50+ accounts is almost certainly a fraud device.
tx_dev = tx[tx["device_str"] != ""]
if len(tx_dev) > 0:
    device_acct_count = (tx_dev.groupby("device_str")["card1"]
                               .nunique()
                               .rename("dev_acct_cnt")
                               .reset_index())
    tx = tx.merge(device_acct_count, on="device_str", how="left")
    tx["dev_acct_cnt"] = tx["dev_acct_cnt"].fillna(1)
    max_cnt = np.percentile(tx["dev_acct_cnt"].values, 99)
    f12_cross = np.clip(tx["dev_acct_cnt"].values / (max_cnt + 1e-9), 0.0, 1.0).astype(np.float32)
else:
    f12_cross = np.zeros(len(tx), dtype=np.float32)
print(f"  F12 cross_device_accounts mean={f12_cross.mean():.3f}")



  F12 cross_device_accounts mean=0.118


In [14]:
# Newer accounts (fewer prior transactions) are higher risk — attackers
# open new accounts or use stolen credentials with thin history.
# 1/(1+txn_seq): first txn=1.0, 10th txn=0.09, 50th txn=0.02
f13_age = (1.0 / (1.0 + tx["txn_seq"].fillna(0).values)).astype(np.float32)
print(f"  F13 account_age_score  mean={f13_age.mean():.3f}")



  F13 account_age_score  mean=0.056


In [15]:
FEATURE_NAMES = [
    "amount_z_score",       # 0 — deviation from account mean
    "merchant_novelty",     # 1 — new merchant category
    "geo_displacement",     # 2 — location shift
    "hour_deviation",       # 3 — unusual time
    "device_novelty",       # 4 — new device
    "velocity_ratio",       # 5 — transaction rate spike
    "behavioral_drift",     # 6 — AE reconstruction error
    "p1_risk_score",        # 7 — Phase 1 ensemble score (patched by NB08)
    "amount_percentile",    # 8 — rank in account's amount history
    "time_gap_norm",        # 9 — gap since last transaction
    "merchant_fraud_rate",  # 10 — global fraud rate at this merchant type
    "cross_device_accounts",# 11 — device shared across many accounts
    "account_age_score",    # 12 — account history depth
]

X_fusion = np.column_stack([
    f1_amount_z, f2_merchant, f3_geo, f4_hour,
    f5_device, f6_vel, f7_drift, f8_p1,
    f9_amt_pct, f10_gap, f11_mfr, f12_cross, f13_age,
]).astype(np.float32)

X_fusion = np.nan_to_num(X_fusion, nan=0.0)

print(f"\n[NB07] Feature matrix: {X_fusion.shape}  (was 590540×8, now 590540×13)")
print("  Feature stats (mean / std / corr-with-label):")
y_tmp = tx["isFraud"].fillna(0).values

for i, name in enumerate(FEATURE_NAMES):
    col  = X_fusion[:, i]
    corr = float(np.corrcoef(col, y_tmp)[0,1]) if np.std(col) > 0 else 0.0
    print(f"  {name:<25}  mean={col.mean():.3f}  std={col.std():.3f}  "
          f"corr={corr:+.4f}")




[NB07] Feature matrix: (590540, 13)  (was 590540×8, now 590540×13)
  Feature stats (mean / std / corr-with-label):
  amount_z_score             mean=0.026  std=1.203  corr=+0.0254
  merchant_novelty           mean=0.231  std=0.298  corr=+0.0144
  geo_displacement           mean=0.097  std=0.147  corr=-0.0088
  hour_deviation             mean=0.912  std=0.162  corr=+0.0080
  device_novelty             mean=0.444  std=0.217  corr=-0.0524
  velocity_ratio             mean=1.595  std=2.747  corr=+0.0470
  behavioral_drift           mean=0.236  std=0.274  corr=+0.1517
  p1_risk_score              mean=0.500  std=0.000  corr=+0.0000
  amount_percentile          mean=0.529  std=0.290  corr=+0.0004
  time_gap_norm              mean=0.243  std=0.260  corr=-0.0354
  merchant_fraud_rate        mean=0.035  std=0.031  corr=+0.1684
  cross_device_accounts      mean=0.118  std=0.295  corr=+0.0526
  account_age_score          mean=0.056  std=0.167  corr=-0.0161


In [16]:
# Standard approach (isFraud):
#   Only the confirmed fraud transactions are positive.
#   All probe/takeover transactions before first fraud = labeled 0.
#   Model cannot learn the pre-fraud behavioral shift.
#
# Industry approach (account_compromised):
#   Mark transactions 24h BEFORE first confirmed fraud as positive.
#   These are the probe transactions during the takeover window.
#   Model learns what accounts look like DURING a compromise.
print("\n[NB07] Building account-compromise window labels...")
print("  Standard isFraud positives: ", int(tx["isFraud"].sum()), "(transaction-level)")

# Find first fraud TransactionDT per account
first_fraud = (tx[tx["isFraud"] == 1]
               .groupby("card1")["TransactionDT"]
               .min()
               .rename("first_fraud_dt")
               .reset_index())
tx = tx.merge(first_fraud, on="card1", how="left")

WINDOW_SECONDS = 24 * 3600   # 24h compromise window before first fraud

# Positive label = confirmed fraud OR within the compromise window
tx["account_compromised"] = (
    (tx["isFraud"] == 1) |
    (
        tx["first_fraud_dt"].notna() &
        (tx["TransactionDT"] >= tx["first_fraud_dt"] - WINDOW_SECONDS) &
        (tx["TransactionDT"] <  tx["first_fraud_dt"]) &
        (tx["isFraud"] == 0)
    )
).astype(np.float32)

y_fusion    = tx["account_compromised"].values.astype(np.float32)
account_ids = tx["card1"].fillna(-1).values.astype(np.int64)

print(f"  Window labels positives  : {int(y_fusion.sum()):,}  "
      f"({100*y_fusion.mean():.2f}%)  ← includes pre-fraud probe window")
print(f"  isFraud positives        : {int(tx['isFraud'].sum()):,}  "
      f"({100*tx['isFraud'].mean():.2f}%)  ← transaction-level only")
print(f"  New positives added      : "
      f"{int(y_fusion.sum()-tx['isFraud'].sum()):,}  (probe window transactions)")




[NB07] Building account-compromise window labels...
  Standard isFraud positives:  20663 (transaction-level)
  Window labels positives  : 23,400  (3.96%)  ← includes pre-fraud probe window
  isFraud positives        : 20,663  (3.50%)  ← transaction-level only
  New positives added      : 2,737  (probe window transactions)


In [18]:
print("\n[NB07] Saving outputs...")
np.save(os.path.join(DATA_DIR, "contextual_features.npy"),  X_fusion)
np.save(os.path.join(DATA_DIR, "fusion_labels.npy"),         y_fusion)
np.save(os.path.join(DATA_DIR, "fusion_account_ids.npy"),    account_ids)

with open(os.path.join(DATA_DIR, "fusion_feature_names.json"), "w") as f:
    json.dump(FEATURE_NAMES, f, indent=2)

print(f"  contextual_features.npy  → {X_fusion.shape}")
print(f"  fusion_labels.npy        → positive={int(y_fusion.sum()):,} ({100*y_fusion.mean():.2f}%)")
print(f"  fusion_feature_names.json → {len(FEATURE_NAMES)} features")

# NaN check
nan_total = int(np.isnan(X_fusion).sum())
print(f"  Total NaN in matrix: {nan_total}  {'✓' if nan_total == 0 else '← FIX NEEDED'}")

# Correlation table
print("\n[NB07] Feature–label correlations:")
for i, name in enumerate(FEATURE_NAMES):
    col  = X_fusion[:, i]
    corr = float(np.corrcoef(col, y_fusion)[0,1]) if np.std(col) > 0 else 0.0
    bar  = "█" * int(abs(corr) * 50)
    sign = "+" if corr >= 0 else "-"
    print(f"  {name:<25} {sign}{abs(corr):.4f}  {bar}")

print(f"\n[NB07] ✓ Done — 13 features, account-window labels")
print(f"[NB07] NEXT → python notebooks/08_fusion_model_training.py")



[NB07] Saving outputs...
  contextual_features.npy  → (590540, 13)
  fusion_labels.npy        → positive=23,400 (3.96%)
  fusion_feature_names.json → 13 features
  Total NaN in matrix: 0  ✓

[NB07] Feature–label correlations:
  amount_z_score            +0.0248  █
  merchant_novelty          +0.0148  
  geo_displacement          -0.0045  
  hour_deviation            +0.0036  
  device_novelty            -0.0480  ██
  velocity_ratio            +0.0529  ██
  behavioral_drift          +0.1405  ███████
  p1_risk_score             +0.0000  
  amount_percentile         +0.0012  
  time_gap_norm             -0.0420  ██
  merchant_fraud_rate       +0.1579  ███████
  cross_device_accounts     +0.0516  ██
  account_age_score         -0.0127  

[NB07] ✓ Done — 13 features, account-window labels
[NB07] NEXT → python notebooks/08_fusion_model_training.py
